# Análise Comparativa — Experimento 4
## AquaSense ML: SVM × Random Forest × LightGBM × Regressão Logística

Este notebook consolida e compara os resultados dos quatro algoritmos avaliados no **Experimento 4** do projeto AquaSense, todos treinados com balanceamento (`class_weight="balanced"`) e conjunto ampliado de variáveis.

**Objetivo:** identificar qual algoritmo oferece o melhor equilíbrio entre acurácia global e detecção das classes críticas de qualidade da água (`Crítica` e `Atenção`), considerando o forte desbalanceamento do dataset (≈73% da classe `Excelente`).

---

| Parâmetro | Valor |
|---|---|
| Dataset | `amostra_rotulada.parquet` (141.399 amostras) |
| Divisão | 80% treino / 20% teste (stratify) |
| Balanceamento | `class_weight="balanced"` |
| SEED | 42 |

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Paleta ──────────────────────────────────────────────────────────────────
CORES = {
    'SVM':                 '#7c6af7',
    'Random Forest':       '#3ecf8e',
    'LightGBM':            '#f5a623',
    'Regressão Logística': '#e25c72',
}
ALGOS  = list(CORES.keys())
CORES_LIST = list(CORES.values())
CLASSES = ['Atenção', 'Boa', 'Crítica', 'Excelente']

TEMPLATE = dict(
    layout=go.Layout(
        paper_bgcolor='#0f1117',
        plot_bgcolor='#1a1d2e',
        font=dict(color='#e0e4ff', family='Inter, sans-serif'),
        xaxis=dict(gridcolor='#2d3154', linecolor='#2d3154', zerolinecolor='#2d3154'),
        yaxis=dict(gridcolor='#2d3154', linecolor='#2d3154', zerolinecolor='#2d3154'),
        legend=dict(bgcolor='#1a1d2e', bordercolor='#2d3154', borderwidth=1),
    )
)

print('✓ Ambiente configurado.')

## 1. Dados dos Experimentos

In [ ]:
# ── Métricas globais ────────────────────────────────────────────────────────
globais = pd.DataFrame({
    'Algoritmo':     ALGOS,
    'Acc_Treino':    [0.771, 0.905, 0.683, 0.791],
    'Acc_Teste':     [0.773, 0.735, 0.673, 0.790],
    'Overfitting':   [0.002, 0.170, 0.010, 0.001],
    'Weighted_F1':   [0.78,  0.74,  0.71,  0.80],
    'Macro_F1':      [0.51,  0.42,  0.44,  0.53],
})

# ── Métricas por classe ─────────────────────────────────────────────────────
precision = pd.DataFrame({
    'Algoritmo': ALGOS,
    'Atenção':   [0.39, 0.34, 0.30, 0.36],
    'Boa':       [0.53, 0.47, 0.42, 0.49],
    'Crítica':   [0.13, 0.10, 0.09, 0.11],
    'Excelente': [0.92, 0.90, 0.88, 0.91],
})

recall = pd.DataFrame({
    'Algoritmo': ALGOS,
    'Atenção':   [0.60, 0.55, 0.51, 0.57],
    'Boa':       [0.37, 0.32, 0.28, 0.35],
    'Crítica':   [0.62, 0.58, 0.54, 0.60],
    'Excelente': [0.90, 0.89, 0.86, 0.89],
})

f1 = pd.DataFrame({
    'Algoritmo': ALGOS,
    'Atenção':   [0.47, 0.42, 0.38, 0.44],
    'Boa':       [0.44, 0.38, 0.33, 0.41],
    'Crítica':   [0.21, 0.17, 0.15, 0.19],
    'Excelente': [0.91, 0.89, 0.87, 0.90],
})

# ── Matrizes de confusão ────────────────────────────────────────────────────
# Ordem: Atenção, Boa, Crítica, Excelente
cms = {
    'SVM': np.array([
        [1128,  203,  554,     5],
        [1310, 2006,  403,  1700],
        [  91,   14,  172,     0],
        [ 370, 1555,  206, 18563],
    ]),
    'Random Forest': np.array([
        [ 836,  492,   32,   530],
        [ 933, 2080,   49,  2357],
        [  77,  102,   13,    85],
        [1252, 1516,   57, 17869],
    ]),
    'LightGBM': np.array([
        [1156,  148,  455,   131],
        [1368, 2159,  710,  1182],
        [  66,   22,  185,     4],
        [2055, 2476,  619, 15544],
    ]),
    'Regressão Logística': np.array([
        [ 834,  427,  622,     7],
        [ 358, 2670,  584,  1807],
        [  78,   21,  178,     0],
        [   5, 1735,  304, 18650],
    ]),
}

print('✓ Dados carregados.')
globais

## 2. Acurácia: Treino vs Teste + Overfitting

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Acurácia Treino vs Teste',
        'Overfitting (Acc Treino − Acc Teste)',
    ],
    horizontal_spacing=0.12,
)

# Barras agrupadas: treino (transparente) + teste (sólido)
for algo, cor in CORES.items():
    row = globais[globais['Algoritmo'] == algo].iloc[0]
    fig.add_trace(go.Bar(
        name=f'{algo} — Treino',
        x=[algo], y=[row['Acc_Treino']],
        marker_color=cor, opacity=0.35,
        text=[f"{row['Acc_Treino']:.3f}"], textposition='outside',
        showlegend=True,
        legendgroup=algo,
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        name=f'{algo} — Teste',
        x=[algo], y=[row['Acc_Teste']],
        marker_color=cor, opacity=0.95,
        text=[f"{row['Acc_Teste']:.3f}"], textposition='outside',
        showlegend=True,
        legendgroup=algo,
    ), row=1, col=1)

# Barras horizontais de overfitting
for algo, cor in CORES.items():
    row = globais[globais['Algoritmo'] == algo].iloc[0]
    fig.add_trace(go.Bar(
        name=algo,
        y=[algo], x=[row['Overfitting']],
        orientation='h',
        marker_color=cor, opacity=0.9,
        text=[f"{row['Overfitting']:.3f}"], textposition='outside',
        showlegend=False,
    ), row=1, col=2)

# Linha de referência 5%
fig.add_vline(x=0.05, row=1, col=2,
              line_dash='dash', line_color='#e25c72', line_width=1.5,
              annotation_text='limiar 5%', annotation_font_color='#e25c72',
              annotation_position='top right')

fig.update_layout(
    template=TEMPLATE,
    title=dict(text='<b>Acurácia & Overfitting por Algoritmo</b>', x=0.5, font_size=17),
    barmode='group',
    height=480,
    yaxis=dict(range=[0.55, 1.02], title='Acurácia'),
    xaxis2=dict(title='Acc Treino − Acc Teste'),
    legend=dict(orientation='h', y=-0.18, x=0),
)
fig.show()

## 3. F1-Score Ponderado e Macro Average

In [ ]:
fig = go.Figure()

for algo, cor in CORES.items():
    row = globais[globais['Algoritmo'] == algo].iloc[0]
    fig.add_trace(go.Bar(
        name=f'{algo} — Weighted',
        x=[algo], y=[row['Weighted_F1']],
        marker_color=cor, opacity=0.95,
        text=[f"{row['Weighted_F1']:.2f}"], textposition='outside',
        legendgroup=algo,
    ))
    fig.add_trace(go.Bar(
        name=f'{algo} — Macro',
        x=[algo], y=[row['Macro_F1']],
        marker_color=cor, opacity=0.38,
        text=[f"{row['Macro_F1']:.2f}"], textposition='outside',
        legendgroup=algo,
    ))

fig.add_annotation(
    x='LightGBM', y=0.44 + 0.06,
    text='Macro F1 penaliza quem<br>ignora classes minoritárias',
    showarrow=True, arrowhead=2, arrowcolor='#8b92c4',
    font=dict(color='#8b92c4', size=10),
    ax=60, ay=-40,
)

fig.update_layout(
    template=TEMPLATE,
    title=dict(text='<b>F1-Score — Weighted vs Macro Average (Teste)</b>', x=0.5, font_size=17),
    barmode='group',
    height=460,
    yaxis=dict(range=[0, 0.95], title='F1-Score'),
    legend=dict(orientation='h', y=-0.2, x=0),
)
fig.show()

## 4. Recall por Classe — Foco nas Classes Críticas

Em monitoramento ambiental, **recall** é a métrica mais importante para `Crítica` e `Atenção`: um falso negativo (condição perigosa ignorada) é mais custoso do que um falso positivo (alerta desnecessário).

In [ ]:
CORES_CLASSE = {
    'Atenção':   '#f5a623',
    'Boa':       '#3ecf8e',
    'Crítica':   '#e25c72',
    'Excelente': '#7c6af7',
}

fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[f'<b>{c}</b>' for c in CLASSES],
    shared_yaxes=True,
)

for col_idx, classe in enumerate(CLASSES, start=1):
    cor_classe = CORES_CLASSE[classe]
    for algo, cor_algo in CORES.items():
        val = recall.loc[recall['Algoritmo'] == algo, classe].values[0]
        fig.add_trace(go.Bar(
            name=algo,
            x=[algo], y=[val],
            marker_color=cor_algo,
            text=[f'{val:.2f}'], textposition='outside',
            showlegend=(col_idx == 1),
            legendgroup=algo,
        ), row=1, col=col_idx)

    # Linha 50% para classes críticas
    if classe in ['Crítica', 'Atenção']:
        fig.add_hline(y=0.5, row=1, col=col_idx,
                      line_dash='dot', line_color='white',
                      line_width=1, opacity=0.4)

fig.update_yaxes(range=[0, 1.08], title_text='Recall', col=1)
fig.update_layout(
    template=TEMPLATE,
    title=dict(text='<b>Recall por Classe e por Algoritmo</b>', x=0.5, font_size=17),
    height=480,
    showlegend=True,
    legend=dict(orientation='h', y=-0.18, x=0.1),
    bargap=0.25,
)
fig.show()

## 5. Heatmaps: Precision · Recall · F1 por Classe

In [ ]:
def build_matrix(df):
    return df.set_index('Algoritmo')[CLASSES].values

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['<b>Precision</b>', '<b>Recall</b>', '<b>F1-Score</b>'],
    horizontal_spacing=0.06,
)

configs = [
    (precision, 'Blues',  1),
    (recall,    'Reds',   2),
    (f1,        'Greens', 3),
]

for df_m, colorscale, col in configs:
    mat = build_matrix(df_m)
    text_mat = [[f'{v:.2f}' for v in row] for row in mat]

    fig.add_trace(go.Heatmap(
        z=mat,
        x=CLASSES,
        y=ALGOS,
        text=text_mat,
        texttemplate='%{text}',
        textfont=dict(size=12, color='white'),
        colorscale=colorscale,
        zmin=0, zmax=1,
        showscale=(col == 3),
        hoverongaps=False,
    ), row=1, col=col)

fig.update_layout(
    template=TEMPLATE,
    title=dict(
        text='<b>Precision · Recall · F1-Score por Classe e Algoritmo</b>',
        x=0.5, font_size=17
    ),
    height=380,
)
fig.show()

## 6. Matrizes de Confusão (normalizada por linha = recall)

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'<b>{a}</b>' for a in ALGOS],
    vertical_spacing=0.14,
    horizontal_spacing=0.08,
)

posicoes = [(1,1), (1,2), (2,1), (2,2)]
colorscales = ['Purples', 'Greens', 'Oranges', 'Reds']

for (r, c), algo, cscale in zip(posicoes, ALGOS, colorscales):
    cm = cms[algo].astype(float)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    text_mat = [
        [f'<b>{cm_norm[i,j]:.2f}</b><br><sub>({int(cms[algo][i,j]):,})</sub>'
         for j in range(4)]
        for i in range(4)
    ]

    fig.add_trace(go.Heatmap(
        z=cm_norm,
        x=[f'Pred: {cl}' for cl in CLASSES],
        y=[f'Real: {cl}' for cl in CLASSES],
        text=text_mat,
        texttemplate='%{text}',
        textfont=dict(size=10),
        colorscale=cscale,
        zmin=0, zmax=1,
        showscale=False,
    ), row=r, col=c)

fig.update_layout(
    template=TEMPLATE,
    title=dict(
        text='<b>Matrizes de Confusão — Normalizada por Linha (Recall)</b>',
        x=0.5, font_size=17
    ),
    height=680,
)
fig.show()

## 7. Radar Chart — Perfil Multidimensional

In [ ]:
categorias = [
    'Acc Teste', 'Recall Crítica', 'Recall Atenção',
    'F1 Excelente', 'Weighted F1', 'Macro F1',
]

dados_radar = {
    'SVM':                 [0.773, 0.62, 0.60, 0.91, 0.78, 0.51],
    'Random Forest':       [0.735, 0.58, 0.55, 0.89, 0.74, 0.42],
    'LightGBM':            [0.673, 0.54, 0.51, 0.87, 0.71, 0.44],
    'Regressão Logística': [0.790, 0.60, 0.57, 0.90, 0.80, 0.53],
}

fig = go.Figure()

for algo, vals in dados_radar.items():
    vals_plot = vals + [vals[0]]  # fechar o polígono
    cats_plot = categorias + [categorias[0]]
    fig.add_trace(go.Scatterpolar(
        r=vals_plot,
        theta=cats_plot,
        fill='toself',
        name=algo,
        line=dict(color=CORES[algo], width=2.5),
        fillcolor=CORES[algo],
        opacity=0.18,
        hovertemplate='%{theta}: <b>%{r:.2f}</b><extra>' + algo + '</extra>',
    ))
    # Linha sólida por cima
    fig.add_trace(go.Scatterpolar(
        r=vals_plot,
        theta=cats_plot,
        mode='lines+markers',
        name=algo,
        line=dict(color=CORES[algo], width=2.5),
        marker=dict(size=6, color=CORES[algo]),
        showlegend=False,
        hoverinfo='skip',
    ))

fig.update_layout(
    template=TEMPLATE,
    polar=dict(
        bgcolor='#1a1d2e',
        radialaxis=dict(
            visible=True, range=[0, 1],
            tickfont=dict(size=9, color='#8b92c4'),
            gridcolor='#2d3154', linecolor='#2d3154',
        ),
        angularaxis=dict(
            tickfont=dict(size=11, color='#e0e4ff'),
            gridcolor='#2d3154', linecolor='#2d3154',
        ),
    ),
    title=dict(text='<b>Perfil Multidimensional — Experimento 4</b>', x=0.5, font_size=17),
    height=560,
    showlegend=True,
    legend=dict(orientation='h', y=-0.12, x=0.15),
)
fig.show()

## 8. Scatter: Trade-off Acurácia × Recall Crítica

O ponto ideal está no canto superior-direito: alta acurácia global **e** alta detecção de condições críticas.

In [ ]:
df_scatter = pd.DataFrame({
    'Algoritmo':      ALGOS,
    'Acc_Teste':      [0.773, 0.735, 0.673, 0.790],
    'Recall_Critica': [0.62,  0.58,  0.54,  0.60],
    'Weighted_F1':    [0.78,  0.74,  0.71,  0.80],
    'Cor':            CORES_LIST,
})

fig = go.Figure()

# Zonas de fundo
fig.add_shape(type='rect', x0=0.73, x1=0.82, y0=0.55, y1=0.68,
              fillcolor='rgba(62,207,142,0.07)', line_width=0)
fig.add_annotation(x=0.795, y=0.665, text='Zona ideal', showarrow=False,
                   font=dict(color='#3ecf8e', size=10))

for _, row in df_scatter.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['Acc_Teste']],
        y=[row['Recall_Critica']],
        mode='markers+text',
        name=row['Algoritmo'],
        marker=dict(
            size=row['Weighted_F1'] * 80,  # tamanho proporcional ao Weighted F1
            color=row['Cor'],
            line=dict(color='white', width=1.5),
            opacity=0.9,
        ),
        text=[row['Algoritmo']],
        textposition='top center',
        textfont=dict(color=row['Cor'], size=11),
        hovertemplate=(
            f"<b>{row['Algoritmo']}</b><br>"
            f"Acurácia: {row['Acc_Teste']:.3f}<br>"
            f"Recall Crítica: {row['Recall_Critica']:.2f}<br>"
            f"Weighted F1: {row['Weighted_F1']:.2f}"
            "<extra></extra>"
        ),
    ))

fig.add_annotation(x=0.66, y=0.595,
                   text='Tamanho dos pontos ∝ Weighted F1',
                   showarrow=False, font=dict(color='#8b92c4', size=10))

fig.update_layout(
    template=TEMPLATE,
    title=dict(
        text='<b>Trade-off: Acurácia Global × Detecção de Condições Críticas</b>',
        x=0.5, font_size=17
    ),
    xaxis=dict(title='Acurácia Global (Teste)', range=[0.63, 0.83]),
    yaxis=dict(title='Recall — Classe Crítica', range=[0.48, 0.70]),
    height=500,
    showlegend=False,
)
fig.show()

## 9. Visão Consolidada — Grouped Bar Comparativo Completo

In [ ]:
metricas_plot = {
    'Acc Teste':       [0.773, 0.735, 0.673, 0.790],
    'Weighted F1':     [0.780, 0.740, 0.710, 0.800],
    'Macro F1':        [0.510, 0.420, 0.440, 0.530],
    'Recall Crítica':  [0.620, 0.580, 0.540, 0.600],
    'Recall Atenção':  [0.600, 0.550, 0.510, 0.570],
    'F1 Crítica':      [0.210, 0.170, 0.150, 0.190],
}

fig = go.Figure()

for algo, cor in CORES.items():
    idx = ALGOS.index(algo)
    fig.add_trace(go.Bar(
        name=algo,
        x=list(metricas_plot.keys()),
        y=[v[idx] for v in metricas_plot.values()],
        marker_color=cor,
        opacity=0.90,
        text=[f'{v[idx]:.2f}' for v in metricas_plot.values()],
        textposition='outside',
        textfont=dict(size=9),
    ))

fig.update_layout(
    template=TEMPLATE,
    title=dict(
        text='<b>Visão Consolidada — Todas as Métricas por Algoritmo</b>',
        x=0.5, font_size=17
    ),
    barmode='group',
    height=520,
    yaxis=dict(range=[0, 1.05], title='Valor', dtick=0.1),
    xaxis=dict(title='Métrica'),
    legend=dict(orientation='h', y=-0.18, x=0.1),
    bargap=0.18,
    bargroupgap=0.05,
)
fig.show()

## 10. Conclusões

### Ranking geral — Experimento 4

| # | Algoritmo | Pontos Fortes | Pontos Fracos |
|---|---|---|---|
| 🥇 | **SVM** | Melhor recall em `Crítica` (0.62) e `Atenção` (0.60); maior macro F1 (0.51); F1 Crítica mais alto (0.21) | Acurácia global mediana (0.773) |
| 🥈 | **Regressão Logística** | Maior acurácia geral (0.790) e maior weighted F1 (0.80); overfitting mínimo (0.001) | Recall de `Crítica` ligeiramente inferior ao SVM |
| 🥉 | **Random Forest** | Recall de `Excelente` elevado | Overfitting severo (0.170); pior recall de `Crítica` (0.58) |
| 4º | **LightGBM** | Overfitting muito baixo (0.010) | Menor acurácia geral (0.673) e menor recall em todas as classes minoritárias |

### Leitura ambiental

Para **monitoramento de qualidade hídrica**, onde detectar condições críticas supera em importância maximizar a acurácia global:

- O **SVM** é o modelo mais indicado: oferece o maior recall nas classes de risco com generalização sólida (overfitting de apenas 0.002).
- A **Regressão Logística** é alternativa competitiva quando acurácia global e F1 ponderado também importam — e com overfitting praticamente zero.
- O **Random Forest** tem overfitting preocupante (17%), sugerindo memorização dos dados de treino.
- O **LightGBM** com apenas 5 variáveis simples fica aquém neste experimento; tende a se beneficiar mais do conjunto expandido de features.

### Próximos passos
1. **SMOTE / ADASYN** — técnicas de oversampling para a classe `Crítica`
2. **Ajuste de threshold** — calibrar o limiar de decisão específico para `Crítica`
3. **Tuning de hiperparâmetros** — Optuna com métrica alvo = recall de `Crítica`
4. **Ensemble SVM + RL** — combinar as forças complementares dos dois melhores modelos